In [ ]:
#coding=gbk
import math
import random
import matplotlib.pyplot as plt
import time
import numpy as np
from decimal import *
getcontext().prec=36

class T_kernel_SGD():
    def __init__(self,s,Q,sigma,theta,loss):
        self.s = s
        self.Ln = 0
        self.fn = [[0,0]]
        self.fn_average = [[0,0]]
        self.theta = theta
        self.gamma0 = 1
        self.t = 0
        self.epoch = 1
        self.Q = Q
        self.loss = loss
        self.sigma = sigma
        self.sqrt_2 = math.sqrt(2)

    def Computation_Yn1(self,x):
        Yn_11 = self.sqrt_2
        Yn1 = self.sqrt_2*x
        if self.Ln == 0:
            self.Yn1_List = [Yn_11]
        elif self.Ln >= 1:
            self.Yn1_List = [Yn_11,Yn1]
            for i in range(1,self.Ln):
                Yn = 2*x*self.Yn1_List[-1]-self.Yn1_List[-2]
                self.Yn1_List.append(Yn)
        self.Yn1_List[0] = 1

    def Computation_Yn2(self,x1,x2):
        Yn_12 = self.sqrt_2*x2
        Yn2 = 2*x1*self.sqrt_2*x2
        if self.Ln == 0:
            self.Yn2_List = [0]
        elif self.Ln == 1:
            self.Yn2_List = [0,Yn_12]
        elif self.Ln >= 2:
            self.Yn2_List = [0,Yn_12,Yn2]
            for i in range(2,self.Ln):
                Yn = 2*x1*self.Yn2_List[-1]-self.Yn2_List[-2]
                self.Yn2_List.append(Yn)                  
                
        
    def Computation_fn(self,X):
        self.Computation_Yn1(X[0])
        self.Computation_Yn2(X[0],X[1])
        S = 0
        i = 0
        for a in self.fn:
            S += a[0]*self.Yn1_List[i]+a[1]*self.Yn2_List[i]
            i += 1
        return S

    def Computation_fn_average(self,X):
        self.Computation_Yn1(X[0])
        self.Computation_Yn2(X[0],X[1])
        S = 0
        i = 0
        for a in self.fn_average:
            S += a[0]*self.Yn1_List[i]+a[1]*self.Yn2_List[i]
            i += 1
        return S
    
    def derivative_loss(self,X,Y):
        if self.loss == 'least square':
            derivative = (self.Computation_fn(X)-Y)
        elif self.loss == 'logistic':
            term = math.exp(Y*self.Computation_fn(X))
            derivative = - Y / (1+term)
        elif self.loss == 'Cauchy':
            term = self.Computation_fn(X)-Y
            derivative = term/(self.sigma**2+term**2/2)
        elif self.loss == 'Huber':
            term = self.Computation_fn(X)-Y
            derivative = term/math.sqrt(term**2+1)
        elif self.loss == 'Welsch':
            term = self.Computation_fn(X)-Y
            derivative = math.exp(-term**2/(2*self.sigma**2))*(term/self.sigma**2)
        return derivative
    
    def iteration(self,X,Y):
        coe = -self.gamma0*((self.epoch)**(-self.t))*self.derivative_loss(X,Y)
        if self.Ln*2+1 < self.epoch**self.theta:
            self.Ln += 1
            self.fn.append([0,0])
            self.fn_average.append([0,0])
        self.Computation_Yn1(X[0])
        self.Computation_Yn2(X[0],X[1])
        Ln_List = list(range(self.Ln+1))
        #print(self.Yn1_List,self.fn,self.Ln)
        self.fn[0] = [self.fn[0][0]+coe*self.Yn1_List[0],self.fn[0][1]+coe*self.Yn2_List[0]]
        self.fn[1:] = [[a[0]+coe*(2*k)**(-2*self.s)*self.Yn1_List[k],a[1]+coe*(2*k)**(-2*self.s)*self.Yn2_List[k]] for k,a in zip(Ln_List[1:],self.fn[1:])]
        self.fn_norm = math.sqrt(self.fn[0][0]**2+self.fn[0][1]**2+sum([(a[0]**2+a[1]**2)*((2*k)**(2*self.s)) for k,a in zip(Ln_List[1:],self.fn[1:])]))
        if self.fn_norm > self.Q:
            zoom = self.Q/self.fn_norm
            self.fn = [[a[0]*zoom,a[1]*zoom] for a in self.fn]
        if self.epoch > self.Sum_epoch/2:
            self.fn_average = [[(a[0]+self.average_coe*b[0])/(self.average_coe+1),(a[1]+self.average_coe*b[1])/(self.average_coe+1)] for a,b in zip(self.fn,self.fn_average)]
            self.average_coe += 1

    def grho_cal(self,theta):
        t = theta/(2*math.pi)
        return (t**2-t+1/6)/5 #(t**4-2*(t**3)+t**2-1/30)/10
        
    def construct_data(self):
        theta = 2*math.pi*random.random()
        Y = self.grho_cal(theta)
        Z = Y + random.uniform(-0.2,0.2)
        return [[np.cos(theta),np.sin(theta)],Z,Y,theta]

    def test(self):
        S1 = 0
        S2 = 0
        #end = time.perf_counter()
        T = 2000
        for j in range(T):
            Z = self.construct_data()
            S1 += (self.Computation_fn(Z[0])-Z[2])**2
            S2 += (self.Computation_fn_average(Z[0])-Z[2])**2
        #print('iteration:',i,S1/1000,S2/1000,self.L_n,end-start)
        return [S1/T,S2/T]

    def updating(self,gamma0,t,theta,Sum_epoch,NN,Uni):
        self.gamma0 = gamma0
        self.t = t
        self.theta = theta
        self.Error_fn_ave = 0
        self.Error_fn = 0
        self.Time = 0
        self.Sum_epoch = Sum_epoch
        self.average_coe = 0
        start = time.perf_counter()
        for i in range(1,Sum_epoch+1):
            self.epoch = i
            Z = self.construct_data()
            self.iteration(Z[0],Z[1])
        end = time.perf_counter()
        self.Time = end-start
        S1,S2= self.test()
        self.Error_fn_ave = S2
        self.Error_fn = S1
        Ln_List = list(range(self.Ln+1))
        fn_norm_last = self.fn[0][0]**2+self.fn[0][1]**2+sum([(a[0]**2+a[1]**2)*((2*k)**(2*self.s)) for k,a in zip(Ln_List[1:],self.fn[1:])])
        fn_norm_L2 = self.fn[0][0]**2+self.fn[0][1]**2+sum([(a[0]**2+a[1]**2) for a in self.fn[1:]])
        self.norm_ratio = fn_norm_last/(fn_norm_L2+10**(-8))
        #print(i,self.Error_fn_ave,self.Error_fn,self.Time,self.fn_norm)
        #print(self.Ln,len(self.fn_average))
            #print(i,end-start)
            #self.Error.append(S2)
            #print(self.f_average_n)
            #print('K_n',self.K_n)

        
    
        

In [ ]:
'r change'
N = [int(10**(i/5)) for i in range(1,29)]
s = 1.
r=3/4
gamma0 =4.
Sum_epoch = int(N[-1]+1)
sigma = 1.
Q = 1.
T = 20
r_settings = [
    (1/2, '1_2'),
    (2/3, '2_3'),
    (3/4, '3_4'),
    (1.0, '1'),
    (5/4, '5_4')
]
for r, r_tag in r_settings:
    t = 2*r/(2*r+1)
    theta = 1/(2*s*(2*r+1))
    err_suffix = []
    err_suffix_ave = []
    for i in range(T):
        E_ave = []
        for k in N:
            t_kernel_SGD = T_kernel_SGD(s+10**(-8)*i+10**(-12)*np.log(k),Q,sigma,theta,'Huber')
            t_kernel_SGD.updating(gamma0,t,theta,k,N,'error')
            E_ave.append(t_kernel_SGD.Error_fn_ave)
        err_suffix.append(E_ave[:])
    for i in range(len(err_suffix[-1])):
        S = 0
        for j in range(T):
            S += err_suffix[j][i]
        err_suffix_ave.append(S/T)
    variable_name = 'err_t_kernel_SGD_ave_Huber_B2_r_' + r_tag
    print(variable_name + ' = ' + repr(err_suffix_ave))
    print(' ')

In [ ]:
's change'
N = [int(10**(i/5)) for i in range(1,29)]
t = 3/5
theta = 1/5
gamma0 = 4.
Sum_epoch = int(N[-1]+1)
sigma = 1.
Q = 1.
T = 20
s_settings = [
    (3/4, '3_4'),
    (7/8, '7_8'),
    (1, '1'),
    (9/8, '9_8'),
    (5/4, '5_4')
]
for s, s_tag in s_settings:
    err_suffix = []
    err_suffix_ave = []
    for i in range(T):
        E_ave = []
        for k in N:
            t_kernel_SGD = T_kernel_SGD(s+10**(-8)*i+10**(-12)*np.log(k),Q,sigma,theta,'Huber')
            t_kernel_SGD.updating(gamma0,t,theta,k,N,'error')
            E_ave.append(t_kernel_SGD.Error_fn_ave)
        err_suffix.append(E_ave[:])
    for i in range(len(err_suffix[-1])):
        S = 0
        for j in range(T):
            S += err_suffix[j][i]
        err_suffix_ave.append(S/T)
    variable_name = 'err_t_kernel_SGD_ave_Huber_B2_s_' + s_tag
    print(variable_name + ' = ' + repr(err_suffix_ave))
    print(' ')

In [ ]:
'theta change'
N = [int(10**(i/5)) for i in range(1,29)]
s = 1
t = 3/5
gamma0 = 4.
Sum_epoch = int(N[-1]+1)
sigma = 1.
Q = 1.
T = 20
theta_settings = [
    (1/8, '1_8'),
    (1/6, '1_6'),
    (1/5, '1_5'),
    (1/4, '1_4'),
    (1/3, '1_3')
]
for theta, theta_tag in theta_settings:
    err_suffix = []
    err_suffix_ave = []
    for i in range(T):
        E_ave = []
        for k in N:
            t_kernel_SGD = T_kernel_SGD(s+10**(-8)*i+10**(-12)*np.log(k),Q,sigma,theta,'Huber')
            t_kernel_SGD.updating(gamma0,t,theta,k,N,'error')
            E_ave.append(t_kernel_SGD.Error_fn_ave)
        err_suffix.append(E_ave[:])
    for i in range(len(err_suffix[-1])):
        S = 0
        for j in range(T):
            S += err_suffix[j][i]
        err_suffix_ave.append(S/T)
    variable_name = ('err_t_kernel_SGD_ave_Huber_B2_theta_' + theta_tag)
    print(variable_name + ' = ' + repr(err_suffix_ave))
    print(' ')

In [ ]:
N = [int(10**(i/5)) for i in range(1,29)]
s = 1.
r = 3/4
t = 2*r/(2*r+1)
theta = 1/(2*s*(2*r+1))
gamma0 = 4.
sigma = 1.
T = 20
Q_train_size = 10000
Q_validation_size = 10000
data_generator = T_kernel_SGD(s, 1., sigma, theta, 'Huber')
S_Y = 0
Q_train_seed = 20260829
Q_validation_seed = 20260830
random.seed(Q_train_seed)
np.random.seed(Q_train_seed)
for j in range(Q_train_size):
    Z = data_generator.construct_data()
    S_Y += Z[1]**2
s_Y = math.sqrt(S_Y/Q_train_size)
kappa = math.sqrt(1+math.pi**2/12)
Q0 = s_Y/kappa
Q_candidates = [Q0/3,Q0,3*Q0,9*Q0,27*Q0,81*Q0]
validation_errors = []
for Q_candidate in Q_candidates:
    random.seed(Q_train_seed)
    np.random.seed(Q_train_seed)
    Q_model = T_kernel_SGD(s,Q_candidate,sigma,theta,'Huber')
    Q_model.updating(gamma0,t,theta,Q_train_size,N,'error')
    validation_error = 0
    for j in range(Q_validation_size):
        Z = Q_model.construct_data()
        residual = (Q_model.Computation_fn_average(Z[0]) - Z[1])
        validation_error += (math.sqrt(1+residual**2)-1)
    validation_error /= Q_validation_size
    validation_errors.append(float(validation_error))
best_id = int(np.argmin(validation_errors))
Q_hat = Q_candidates[best_id]

print('s_Y =', s_Y)
print('kappa =', kappa)
print('Q0 =', Q0)
print('Q_candidates =', Q_candidates)
print('validation_errors =', validation_errors)
print('Q_hat =', Q_hat)
print(' ')

Q_settings = [
    (1/2, '1_2'),
    (3/4, '3_4'),
    (1, '1'),
    (5/4, '5_4'),
    (Q_hat, 'estimated')
]

for Q, Q_tag in Q_settings:
    err_suffix = []
    err_suffix_ave = []
    for i in range(T):
        E_ave = []
        for k in N:
            t_kernel_SGD = T_kernel_SGD(s+10**(-8)*i+10**(-12)*np.log(k),Q,sigma,theta,'Huber')
            t_kernel_SGD.updating(gamma0,t,theta,k,N,'error')
            E_ave.append(t_kernel_SGD.Error_fn_ave)
        err_suffix.append(E_ave[:])
    for i in range(len(err_suffix[-1])):
        S = 0
        for j in range(T):
            S += err_suffix[j][i]
        err_suffix_ave.append(S/T)
    variable_name = ('err_t_kernel_SGD_ave_Huber_B2_Q_'+Q_tag)
    print(variable_name +' = ' +repr(err_suffix_ave))
    print(' ')